# Practical LLM Training Runbook

**Goal:** understand what an LLM is being trained to do at each stage, build a small fine-tuning pipeline, and learn when training is justified instead of using an API, prompting, tools, or RAG.

This notebook is intentionally practical. Run the small examples first; treat the larger resource estimates as architecture exercises.

**Stack:** PyTorch + Hugging Face Transformers + Datasets + TRL + PEFT + bitsandbytes.

**Primary references:**
- Transformers fine-tuning: https://huggingface.co/docs/transformers/main/training
- TRL: https://huggingface.co/docs/trl/
- PEFT/LoRA: https://huggingface.co/docs/peft/
- QLoRA: https://arxiv.org/abs/2305.14314
- LoRA: https://arxiv.org/abs/2106.09685
- Instruction tuning survey: https://arxiv.org/abs/2308.10792
- DPO: https://arxiv.org/abs/2305.18290
- Domain-adaptive pretraining: https://arxiv.org/abs/2004.10964
- DeepSpeed ZeRO: https://www.deepspeed.ai/tutorials/zero/
- Megatron Core: https://docs.nvidia.com/megatron-core/developer-guide/latest/get-started/quickstart.html

## 1. The mental model: there are several different things people call training

A useful lifecycle is:

**Pretraining → continued/domain pretraining → supervised/instruction fine-tuning → preference optimization → evaluation → deployment.**

### Pretraining
Start with random or near-random weights and learn next-token prediction over a very large corpus. This creates a general foundation model. Compute, data, distributed systems and data quality dominate the project.

### Continued pretraining / CPT / DAPT
Start from an existing pretrained model and continue the same language-model objective on domain text. This is useful when the model needs to become better adapted to a vocabulary, syntax, style or distribution that is poorly represented in the original training mix.

### SFT / instruction tuning
Train on examples of **input/instruction → desired output**. The objective is still usually next-token loss, but the data teaches the model how you want it to respond.

### Preference optimization
Provide preferred vs rejected answers, or another reward signal, and optimize the model toward the desired behavior. DPO is a common offline approach; RL-style methods such as GRPO are useful when a reward can be computed from sampled outputs.

### Full fine-tuning vs PEFT
These describe **how many parameters are updated**, not what the dataset means. SFT can be full-parameter or LoRA. DPO can be full-parameter or PEFT.


## 2. Dataset: the most important training artifact

A dataset is not simply a pile of text. Define the learning contract first.

| Dataset type | Example | Main objective |
|---|---|---|
| Pretraining corpus | books, code, web, papers | next-token prediction |
| CPT/DAPT corpus | clinical notes, legal text, manuals | domain adaptation |
| SFT dataset | instruction + answer | desired task/behavior |
| Preference dataset | prompt + chosen + rejected | relative preference |
| Reward data | prompt + score/reward | reward modeling or RL |
| Evaluation set | held-out prompts + labels | measure generalization |

### Dataset quality checklist
1. **Correctness:** are answers actually correct?
2. **Coverage:** does the data cover the behavior you need?
3. **Diversity:** are there many near-duplicates?
4. **Consistency:** do examples agree on style and policy?
5. **Difficulty:** does the set include easy, normal and edge cases?
6. **Leakage:** is evaluation information present in training?
7. **Privacy/licensing:** are you allowed to use and train on the data?
8. **Split integrity:** split by user/document/time when random splitting would leak information.
9. **Token budget:** measure tokens, not only row count. A million short rows and a million long documents are radically different workloads.

**Teaching rule:** 5,000 excellent examples can be more useful than 500,000 noisy examples for a narrow SFT task. More data is not automatically better.

In [ ]:
# Install the teaching stack. Restart the kernel if your environment asks you to.
%pip install -q -U transformers datasets accelerate peft trl bitsandbytes sentencepiece evaluate

In [ ]:
import os, math, json, random
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

## 3. Tokenization and the training target

LLMs operate on tokens rather than words. A tokenizer converts text to integer IDs. A causal LM learns to predict the next token:

`x1 x2 x3 ... xN → predict x2 x3 x4 ... xN+1`

For a sequence `I like tea`, the model may be trained on targets corresponding to `like`, `tea`, and an end-of-sequence token. The loss is generally cross-entropy over the target tokens.

For chat/SFT, a conversation is serialized with the model's chat template. Many modern trainers can apply that template automatically. Correct formatting matters: training on a format different from inference can create a silent distribution shift.

In [ ]:
from transformers import AutoTokenizer
model_id = 'Qwen/Qwen3-0.6B'
tokenizer = AutoTokenizer.from_pretrained(model_id)
text = 'Fine-tuning teaches a pretrained model a narrower behavior.'
ids = tokenizer(text).input_ids
print('Text:', text)
print('Token IDs:', ids)
print('Tokens:', tokenizer.convert_ids_to_tokens(ids))
print('Token count:', len(ids))

## 4. Build a tiny instruction dataset

Instruction fine-tuning is a form of supervised fine-tuning (SFT). Each example demonstrates the behavior we want.

For a production dataset, store provenance and versioning metadata even if the trainer only consumes the message fields.

In [ ]:
from datasets import Dataset

examples = [
    {'messages': [
        {'role': 'system', 'content': 'You are a concise teaching assistant.'},
        {'role': 'user', 'content': 'What is an epoch?'},
        {'role': 'assistant', 'content': 'An epoch is one complete pass through the training dataset.'}]},
    {'messages': [
        {'role': 'system', 'content': 'You are a concise teaching assistant.'},
        {'role': 'user', 'content': 'What is LoRA?'},
        {'role': 'assistant', 'content': 'LoRA freezes the base model and learns small low-rank update matrices.'}]},
    {'messages': [
        {'role': 'system', 'content': 'You are a concise teaching assistant.'},
        {'role': 'user', 'content': 'When should I use RAG?'},
        {'role': 'assistant', 'content': 'Use RAG when the answer depends on external or changing documents that should remain outside model weights.'}]},
]
train_ds = Dataset.from_list(examples)
train_ds

## 5. Supervised fine-tuning (SFT)

The simplest modern training recipe is: choose an instruction-tuned or base model → format examples → tokenize/chat-template them → compute next-token loss on the desired completion → update weights.

Hugging Face's current TRL provides `SFTTrainer`; Transformers provides the general `Trainer`. The trainer handles batching, shuffling, forward pass, loss, backpropagation and optimizer steps.

### What SFT is good at
- output format and structure
- task procedure
- classification and extraction patterns
- tone/style
- tool-call conventions
- domain-specific response patterns

### What SFT is not a great substitute for
- a huge, changing knowledge base
- live prices or inventory
- current database state
- authoritative documents that change frequently

Those are often better handled with retrieval, tools, or APIs.

In [ ]:
# Practical SFT example. This is intentionally tiny; increase data/model size only after the pipeline works.
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir='./outputs/qwen3-0.6b-sft',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    logging_steps=1,
    save_strategy='no',
    report_to='none',
)

trainer = SFTTrainer(
    model=model_id,
    train_dataset=train_ds,
    args=sft_config,
    processing_class=tokenizer,
)
# trainer.train()  # Uncomment after confirming your environment has enough memory.

## 6. Full fine-tuning

In full fine-tuning, essentially all model parameters are trainable. This gives the optimizer maximum freedom but creates large memory and compute requirements.

A useful planning approximation for Adam-style mixed-precision training is **~16 bytes per parameter for weights + gradients + optimizer/master state**, before activations, temporary buffers, fragmentation and framework overhead. The exact number varies with optimizer and implementation.

Examples of the state-only lower-bound intuition:
- 1B parameters → ~16 GB
- 7B → ~112 GB
- 13B → ~208 GB
- 70B → ~1.12 TB

This is **not** the GPU requirement. Activation memory and other overhead push peak memory higher. Distributed strategies such as DeepSpeed ZeRO shard optimizer state, gradients and/or parameters across GPUs and can also offload state to CPU/NVMe.

### When full FT makes sense
- you have strong evidence PEFT cannot close the quality gap;
- the adaptation is broad enough to justify changing the whole parameter space;
- you can afford distributed training and experiment cycles;
- you have enough high-quality data to avoid simply overfitting a narrow behavior.

For most first fine-tuning projects, start with LoRA/QLoRA rather than full FT.

## 7. PEFT: LoRA and QLoRA

**PEFT (parameter-efficient fine-tuning)** freezes most or all of the base model and learns a small number of additional parameters.

### LoRA
For a weight matrix `W`, LoRA learns a low-rank update rather than changing every element directly:

`W' = W + B A`

where `A` and `B` have a much smaller rank than the original matrix. The base weights stay frozen. The result is much smaller trainable state and usually much lower storage cost for each task adapter.

### QLoRA
QLoRA combines LoRA with a quantized frozen base model, commonly 4-bit, while gradients flow through the quantized base into the LoRA adapters. The original QLoRA paper demonstrated 65B fine-tuning on a single 48GB GPU under its experimental setup. Modern kernels and models differ, so treat that result as a proof of feasibility, not a universal hardware guarantee.

### Main knobs
- `r`: adapter rank; higher can increase capacity and memory.
- `lora_alpha`: scaling.
- `lora_dropout`: regularization.
- target modules: which linear projections receive adapters.
- learning rate: usually much higher than full FT, but tune empirically.
- sequence length: often one of the biggest memory drivers.
- effective batch size: per-device batch × gradient accumulation × number of devices.


In [ ]:
# QLoRA skeleton. Uncomment the training block after checking GPU memory.
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    bnb_4bit_use_double_quant=True,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

print(lora_config)

## 8. Continued pretraining / domain-adaptive pretraining

Suppose the problem is not 'answer differently' but 'understand this distribution much better.' Examples include a specialized technical vocabulary, domain syntax, or a large corpus of unlabeled text. Continued pretraining can be a better fit than forcing all information into instruction/answer pairs.

A typical sequence is:

**base/instruction model → CPT/DAPT on domain corpus → SFT on task examples → preference optimization if needed.**

Research on domain-adaptive pretraining has found gains from additional in-domain language-model training across multiple domains. The amount of data and compute needed is nevertheless far larger than a small SFT run.

Use CPT when you have a **large, valuable unlabeled corpus** and a measured reason to change the model's internal representation of that domain. Do not use it merely because you have PDFs; retrieval is usually cheaper for document lookup.

## 9. Preference optimization: DPO, reward models and RL-style methods

SFT says: 'this answer is the target.' Preference training says: 'given these alternatives, this one is better.'

A DPO dataset often looks like:

`prompt | chosen response | rejected response`

DPO directly optimizes a preference objective without requiring the classic separate reward-model + PPO pipeline. It is attractive when you can collect reliable pairwise preferences.

### Reward modeling
Train a model to score outputs. The score can then be used by an RL or search procedure. This is useful when a scalar reward is meaningful and can be evaluated consistently.

### GRPO / RL-style post-training
Methods such as GRPO generate multiple candidate completions and optimize using a reward function. They become particularly interesting when correctness can be programmatically checked—for example mathematics, code tests, structured tool outcomes, or agent environments. TRL currently exposes SFT, DPO, reward modeling, GRPO and other trainers.

### Critical warning
Bad preference data creates bad preferences. If annotators systematically reward verbosity, the model can learn verbosity. If the reward is gamed, RL can optimize the metric rather than the real goal.

In [ ]:
# DPO dataset shape example; this is data preparation, not a full training run.
from datasets import Dataset

preference_ds = Dataset.from_list([
    {
        'prompt': 'Explain LoRA in one paragraph.',
        'chosen': 'LoRA freezes the base model and learns small low-rank updates, making adaptation cheaper.',
        'rejected': 'LoRA is a type of database that stores language model prompts.'
    }
])
preference_ds

## 10. Evaluation: never train without a baseline

Before training, freeze an evaluation set and establish a baseline. Compare **the same prompts and metrics** before and after adaptation.

### Recommended evaluation layers
1. **Training loss:** useful for optimization diagnostics, not enough for product quality.
2. **Held-out loss/perplexity:** tests generalization on a language-model objective.
3. **Task metrics:** exact match, F1, accuracy, JSON validity, tool success, code tests, etc.
4. **Human or expert preference:** pairwise comparison is often useful for open-ended quality.
5. **Regression suite:** known failure cases that must not get worse.
6. **Safety/security tests:** prompt injection, data leakage, unsafe outputs, refusal behavior as appropriate.
7. **System metrics:** latency, throughput, GPU utilization, memory, cost/request.

### A practical experiment table

| Version | Method | Data | Task score | Regression | Latency | Cost/request |
|---|---|---|---:|---:|---:|---:|
| API baseline | Claude/ChatGPT/etc. | prompt only | ... | ... | ... | ... |
| RAG baseline | API + retrieval | documents | ... | ... | ... | ... |
| LoRA | open model | SFT | ... | ... | ... | ... |
| QLoRA | quantized open model | SFT | ... | ... | ... | ... |

Do not select a method because its training loss is lower. Select it because it closes the **business/technical metric gap**.

## 11. Memory and resource estimator

The following estimates are deliberately conservative teaching heuristics. Real peak memory depends on architecture, optimizer, precision, sequence length, checkpointing, kernels and framework.

### Full fine-tuning
Approximate state memory: `parameters × 16 bytes`, then add activation/temporary overhead.

### Inference
Inference is much cheaper than training because you do not store gradients and optimizer states. A rough BF16 weight-only estimate is `parameters × 2 bytes`; quantized 4-bit weight-only storage is about `parameters × 0.5 bytes`, plus scales/metadata and KV cache.

### LoRA
Base-model memory remains, but trainable parameter/optimizer memory is much smaller. Sequence length and activations can still dominate.

### QLoRA
The frozen base is quantized, substantially reducing the weight footprint. The adapter and optimizer states remain in higher precision.

In [ ]:
def full_ft_state_gb(params_billion, bytes_per_param=16):
    return params_billion * 1e9 * bytes_per_param / 2**30

def weight_gb(params_billion, bytes_per_param):
    return params_billion * 1e9 * bytes_per_param / 2**30

for p in [0.6, 1, 3, 7, 13, 70]:
    print(f'{p:>5g}B | full-FT state ~ {full_ft_state_gb(p):7.1f} GB | BF16 weights ~ {weight_gb(p,2):7.1f} GB | 4-bit weights ~ {weight_gb(p,0.5):7.1f} GB')

## 12. Cost model

Training cost is approximately:

`GPU-hours × $/GPU-hour + storage + data processing + experiment overhead + engineering + inference lifecycle`

For classroom arithmetic, a RunPod page currently lists examples around **$0.34/hr for RTX 4090, $1.39/hr for A100 80GB and $2.89/hr for H100**. Verify live prices before purchasing because capacity, location and billing mode change.

These numbers make small experiments surprisingly cheap. The expensive part is often not one successful run; it is the series of experiments, evaluation, failed jobs, model serving and engineering required to turn the result into a reliable system.

### Example
8 hours × $1.39/hr ≈ $11.12 compute-only.
12 hours × $2.89/hr ≈ $34.68 compute-only.

Never compare training cost with API token cost using only one training run. Compare **total cost of ownership over the expected traffic and lifetime**.

In [ ]:
def gpu_cost(hours, dollars_per_gpu_hour, gpus=1):
    return hours * dollars_per_gpu_hour * gpus

for name, rate, hours, gpus in [
    ('4090 example', 0.34, 4, 1),
    ('A100 example', 1.39, 8, 1),
    ('H100 example', 2.89, 12, 1),
    ('8×H100 example', 2.89, 24, 8),
]:
    print(f'{name:20s}: ${gpu_cost(hours, rate, gpus):,.2f}')

## 13. When should I use which method?

### Start here

**Q1: Is the problem mostly about current/private knowledge?**
→ Start with RAG/tools/API. Train only if the model must learn a repeatable behavior around that knowledge rather than merely retrieve it.

**Q2: Is the problem mostly behavior, format, style, classification or procedure?**
→ SFT. Start with LoRA/QLoRA for open models.

**Q3: Is the model weak on a domain distribution itself, and do you have lots of unlabeled domain text?**
→ Consider continued/domain-adaptive pretraining, then SFT.

**Q4: Do you have reliable preferred/rejected answers?**
→ Consider DPO or another preference method.

**Q5: Does PEFT consistently fail to meet the target after reasonable tuning?**
→ Test full fine-tuning.

**Q6: Are you trying to create a new general-purpose foundation model?**
→ Pretraining from scratch. Expect cluster-scale infrastructure, huge data, distributed training, evaluation and safety engineering.

## 14. API vs RAG vs fine-tuning: practical problem map

| Problem | Usually start with | Why | Training becomes justified when... |
|---|---|---|---|
| Summarize arbitrary current documents | API + RAG | knowledge is external | the recurring transformation itself needs learned behavior |
| Customer-support tone + workflow | API/SFT | behavior is stable | API prompting cannot reliably hit the measured target |
| Structured JSON extraction | API + schema | modern APIs can enforce structure | open/local model must achieve lower cost/latency/offline requirements |
| Internal policy Q&A | RAG + citations | policy changes | model needs specialized response behavior, not just policy lookup |
| Code generation | strong API + tests | general models are capable | private/offline model or specialized code distribution justifies adaptation |
| Medical/technical terminology | RAG + strong model | facts must remain authoritative | domain distribution itself is consistently limiting performance |
| Tool/agent calling | prompting + tools | behavior can often be specified | stable high-volume tool policy needs learned behavior and API baseline is insufficient |
| Very high request volume | routing/cache/API | operationally simple | measured API cost/latency exceeds economics of owning a model |
| Strict offline/on-prem execution | open-weight local model | deployment constraint | adaptation is needed to reach target quality |
| New foundation capability | pretraining | no suitable base model | no existing model meets the capability/licensing/data requirements |

### What can make training practically mandatory?
There is rarely a mathematical requirement to train an LLM. Training becomes practically necessary when a requirement cannot be satisfied reliably by the available API/open model plus prompting, retrieval, tools, routing and system engineering. Common examples are strict offline deployment combined with a quality target, highly specialized behavior at very high volume, a persistent distribution shift, or a product requirement to own and control the model weights.

## 15. Production failure modes

### Data problems
- duplicated examples overweight a behavior;
- mislabeled answers teach the wrong behavior;
- train/eval leakage makes the model look better than it is;
- inconsistent chat templates create formatting failures;
- sensitive data is accidentally memorized.

### Optimization problems
- learning rate too high → instability/catastrophic forgetting;
- too many epochs on a tiny dataset → memorization/overfitting;
- context length too high → OOM;
- batch too large → OOM;
- LoRA rank too small → insufficient adaptation; too large → unnecessary memory/capacity;
- quantization settings or unsupported kernels → runtime failures or quality changes.

### Product problems
- model benchmark improves but real users do not;
- fine-tuned model knows yesterday's facts but not today's;
- adapter improves one task and regresses general capabilities;
- training cost looks cheap while inference/maintenance dominates;
- a better model was available from an API during the project.

### Operational checklist
Before launch: freeze dataset version, model revision, tokenizer/chat template, training config, evaluation suite and safety checks. Record GPU type, hours, peak memory, throughput and artifact hashes.

## 16. Teaching exercise: choose the method

For each scenario, ask students to choose the **least expensive method likely to satisfy the requirement**, then defend the decision with a measurable experiment.

1. A company has 2 million internal documents that change daily.
2. A classifier must output one of 18 labels and currently misses edge cases.
3. A local/offline assistant must follow a highly specific response protocol.
4. A legal team has 200 GB of unlabeled legal text and wants better domain language understanding.
5. A math model can generate answers, but a verifier can reliably score whether each answer is correct.
6. A startup wants to build a new general-purpose 70B foundation model.

A strong answer should mention the API baseline, RAG/tool baseline where relevant, training-data availability, evaluation metric, latency/cost constraints, licensing/privacy constraints and expected maintenance.

## 17. Final decision tree

```text
START
  |
  +-- Can an existing API/open model + prompt solve it reliably? -- YES --> USE IT
  | NO
  +-- Is the missing information external/current/private? -- YES --> RAG / TOOLS
  | NO
  +-- Is the missing capability mainly behavior/format/task procedure? -- YES --> SFT
  |                                               |
  |                                               +--> LoRA/QLoRA first
  |
  +-- Is the model's domain distribution itself the problem?
  |       |
  |       +--> lots of unlabeled domain text --> CPT/DAPT → SFT
  |
  +-- Do you have preference data? --> DPO / preference optimization
  |
  +-- Does PEFT fail the target after proper experimentation? --> FULL FT
  |
  +-- Need a new foundation model? --> PRETRAIN FROM SCRATCH
```

### The engineering principle

**Train the smallest thing that closes the measured gap.** A fine-tuned model is not automatically better than a frontier API, and a larger training budget is not automatically better than better data or retrieval.